# Baumdaten fürs Klimadashboard

Sehr grobe Annäherung der Werte durch pauschale Faktoren!

In [ ]:
# Fehlende Pakete installieren, ggf. einkommentieren und ausführen

#!python -m pip install geopandas
#!python -m pip install pandas
#!python -m pip install matplotlib
#!python -m pip install dotenv
#!python -m pip install requests

In [ ]:
import requests
import xml.etree.ElementTree as ET

import geopandas as gpd
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
import json


# Place .env file in the same directory as this script and add the following lines:
# WFS_URL=https://your-wfs-url
# USER=api-username
# PASSWORD=api-password
# PROXY=http://your-proxy-url:port (optional)

load_dotenv()
WFS_URL = os.getenv("WFS_URL")
WFS_USER = os.getenv("WFS_USER")
PASSWORD = os.getenv("PASSWORD")
PROXY = os.getenv("PROXY")
proxies = {"http": PROXY, "https": PROXY} if PROXY else False

if not WFS_URL:
    print("Fehler: WFS_URL ist nicht gesetzt.")
if not WFS_USER:
    print("Fehler: WFS_USER ist nicht gesetzt.")
if not PASSWORD:
    print("Fehler: PASSWORD ist nicht gesetzt.")

CURRENT_YEAR = pd.Timestamp.now().year
GEOJSON_FILE = "data/feature_data.geojson"
layer_name = "tree:stamuenster_point"

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": layer_name,
    "outputFormat": "application/json",
}

if not os.path.exists("data"):
  os.mkdir("data")
  print("Folder 'data' created!")

if os.path.exists(GEOJSON_FILE):
    print(f"Datei '{GEOJSON_FILE}' existiert bereits.")
else:
    print(f"Datei wird gespeichert als: {GEOJSON_FILE}")

    session = requests.Session()
    session.auth = (WFS_USER, PASSWORD)
    response = session.get(WFS_URL, params=params, proxies=proxies)
    
    # catch response errors
    if response.status_code != 200:
        print(f"Fehler beim Abrufen der Daten: {response.status_code} - {response.reason}")
        print(response.text)
        exit(1)

    with open(GEOJSON_FILE, "w") as f:
        json.dump(response.json(), f)

# echo filesize of geojson file in megabytes
print(f"Größe der GeoJSON-Datei: {os.path.getsize(GEOJSON_FILE) / (1024 * 1024):.2f} MB")
gdf_orig = gpd.read_file(GEOJSON_FILE)
print("Spalten: ", gdf_orig.columns)


# Ausreisser entfernen

In [ ]:

CHECK_FIELD = "kronendurchmesser"

print("Kronendurchmesser über 50 Meter herausfiltern")
ausreisser = gdf_orig[gdf_orig["kronendurchmesser"] > 50]
ausreisser = ausreisser.sort_values(by=CHECK_FIELD, ascending=False)
print("So viele Ausreißer rausgefiltert: ")
print(ausreisser[["artdeutsch", CHECK_FIELD]].shape[0])
print ("und zwar:")      
print(ausreisser[["artdeutsch", CHECK_FIELD]])

gdf = gdf_orig.drop(ausreisser.index)

# Alter der Bäume in Klassen aufteilen

In [ ]:
ALTERSGRENZEN = [0, 15, 40, 60, 80, 100, np.inf]
ALTERSKLASSEN = [
    "Jungbaum 0-15 Jahre",
    "Heranwachsend 16-40 Jahre",
    "Heranwachsend 41-60 Jahre",
    "Altbaum 61-80 Jahre",
    "Altbaum 81-100 Jahre",
    "Altbaum über 100 Jahre",
]

# Berechnung der Werte fürs Klimadashboard

In [ ]:

# Altersgruppendefinition fürs Klimadashboard
def altersgruppe(alter):
    if alter < 16:
        return "Jung"
    elif alter < 61:
        return "Mittel"
    else:
        return "Alt"

# Formel für: CO2 Bindung
def co2_rate(alter):
    if alter < 16:
        return 10   # kg/Jahr
    elif alter < 61:
        return 25
    else:
        return 40

gdf["pflanzjahr"] = pd.to_numeric(gdf["pflanzjahr"], errors="coerce")
gdf["alter"] = CURRENT_YEAR - gdf["pflanzjahr"]
pflanzungen = {}

# =========================== Berechnungen ===========================

# Kronenschirmfläche = Kreisfläche = π * r²
gdf["kronenschirmflaeche"] = np.pi * (gdf["kronendurchmesser"] / 2) ** 2 / 10000

# CO2 Bindung in kg pro Jahr (Formel siehe Funktion co2_rate)
gdf["co2_t_pro_jahr"] = gdf["alter"].apply(co2_rate) / 1000

# Sauerstoffproduktion
# 32/12 * kg CO2 -> kg O2
# 748 -> Umrechnungsfaktor kg O2 in "Liter gasförmig, (1 bar, 15 °C)
# mal 1000 weil wir das in Tonnen haben, aber für die Formel kg brauchen
gdf["o2_l_gasfoermig_pro_jahr"] = gdf["co2_t_pro_jahr"] * 1000 * (32 / 12) * 748 

# Kühlleistung: 0.15 kW pro m² Kronenschirmfläche
# mal 10000 weil kronenschirmfläche in hektar 
# mal 365*24 für Anzahl Stunden pro Jahr, um von kW auf kWh zu kommen
gdf["kuehlleistung_kwh"] = gdf["kronenschirmflaeche"] * 0.15 * 10000 * 365 * 24

def berechne_jahr(df, jahr):
    temp = df.copy()

    # Alter im betrachteten Jahr
    temp["alter"] = jahr - temp["pflanzjahr"]

    # Bäume, die damals noch nicht gepflanzt waren, ausblenden
    temp = temp[temp["alter"] >= 0]

    # Nur plausible Pflanzjahre für Altersklassen und Gesamtzahl berücksichtigen.
    alter_mit_pflanzjahr = temp.loc[temp["pflanzjahr"] >= 1800, "alter"]
    altersverteilung = pd.cut(
        alter_mit_pflanzjahr, bins=ALTERSGRENZEN, labels=ALTERSKLASSEN,
        include_lowest=True,
    ).value_counts(sort=False)

    # Altersgruppen erstellen
    temp["gruppe"] = temp["alter"].apply(altersgruppe)

    # Hilfsfunktion, um die %-Werte für ein bestimmtes Feld zu berechnen und dem DataFrame hinzuzufügen
    def addFieldsToDataFrame(df, FIELD):
        erg = (
            temp.groupby("gruppe")[FIELD]
            .sum()
            .reindex(["Jung", "Mittel", "Alt"], fill_value=0)
        )
    
        gesamt = erg.sum()
        vals= erg.values
        percent = erg.values / gesamt * 100
        df["gruppe"] = erg.index
        df[f"{FIELD}"] = vals 
        df[f"{FIELD}_prozent"] = percent
        return df

    # Alle Werte des aktuellen Jahres zum DataFrame hinzufügen
    temp_df = pd.DataFrame()
    temp_df["jahr"] = jahr
    temp_df = addFieldsToDataFrame(temp_df, "kronenschirmflaeche")
    temp_df = addFieldsToDataFrame(temp_df, "co2_t_pro_jahr")
    temp_df = addFieldsToDataFrame(temp_df, "o2_l_gasfoermig_pro_jahr")
    temp_df = addFieldsToDataFrame(temp_df, "kuehlleistung_kwh")
    # TODO - Anzahl der Fällungen => hier gibt es noch eine Datenlieferung von 67
    # TODO: Pflanzungen pro Jahr => Alle Bäume mit Alter 0; #TODO diese Daten müssen noch richtig in die Tabelle pro Jahr (unabhängig von der Altersgruppe)
    pflanzungen[jahr] = (temp["alter"] == 0).sum()
    print(f"Anzahl Pflanzungen im Jahr {jahr}: {pflanzungen[jahr]}")
    return temp_df, altersverteilung

startjahr = 2020
endjahr = CURRENT_YEAR

alle = []
altersverteilungen = {}

for jahr in range(startjahr, endjahr + 1):
    jahreswerte, altersverteilungen[jahr] = berechne_jahr(gdf, jahr)
    jahreswerte["jahr"] = jahr
    alle.append(jahreswerte)

# Dataframe erstellen
slider_df = pd.concat(alle, ignore_index=True)

# Runden der Werte auf 2 Nachkommastellen, außer für die Spalten "jahr" und "gruppe"
cols = {col: 2 for col in slider_df.columns if col not in ("jahr", "gruppe")}
slider_df = slider_df.round(cols)

# Daten als CSV speichern
slider_df.to_csv("data/baumkachel_daten.csv", index=False)

# Daten ausgeben
slider_df


# Plotten der Werte 

In [ ]:
altersverteilungen[CURRENT_YEAR].plot(
    kind="barh",
    figsize=(5, 5),
    color=["lightgreen", "orange", "orange", "lightblue", "lightblue", "lightblue"],
    xlabel="Anzahl Bäume",
    ylabel="Altersklasse",
    title=f"Alter der Bäume im Jahr {CURRENT_YEAR}",
)
plt.tight_layout()

def plot_stacked_bar(df, MYVAL):
    pivot_plot = df.pivot(index="jahr", columns="gruppe", values=MYVAL)

    colors = {"Jung": "#4CAF50", "Mittel": "#2196F3", "Alt": "#FF9800"}

    ax = pivot_plot.plot(
        kind="bar",
        stacked=True,
        color=[colors.get(col, "#888888") for col in pivot_plot.columns],
        figsize=(8, 5),
    )

    ax.set_xlabel("Jahr")
    ax.set_ylabel("qm oder kg oder Liter oder kW")
    ax.set_title(f"Gestapelter Anteil der Altersgruppen an der {MYVAL} pro Jahr")
    #ax.set_ylim(0, 100)
    ax.legend(title="Altersgruppe")
    plt.tight_layout()


plot_stacked_bar(slider_df, "kronenschirmflaeche")
plot_stacked_bar(slider_df, "co2_t_pro_jahr")
plot_stacked_bar(slider_df, "o2_l_gasfoermig_pro_jahr")
plot_stacked_bar(slider_df, "kuehlleistung_kwh")




# Output Datei erstellen

In [ ]:
# DATEINAME;RAUM;QUELLE_INSTITUTION;THEMENBEREICH;MERKMAL;ZEIT;WERT;WERTEEINHEIT
# verkehrsmittelwahl-zeitreihe;Münster, Gesamtstadt;Stadt Münster - Amt für Mobilität und Tiefbau;4;Wege/Tag - Kfz;1982;39,20;%
# verkehrsmittelwahl-zeitreihe;Münster, Gesamtstadt;Stadt Münster - Amt für Mobilität und Tiefbau;4;Wege/Tag - ÖV;1982;6,60;%

output_df = []

fields = {
    "kronenschirmflaeche": ["Kronenschirmfläche", "ha"],
    "co2_t_pro_jahr": ["CO2 Bindung", "t"],
    "o2_l_gasfoermig_pro_jahr": ["O2 Gasförmig pro Jahr", "Liter"],
    "kuehlleistung_kwh": ["Kühlleistung", "kWh"]
}

themenbereich = 670
for field in fields:
    themenbereich= themenbereich + 1
    for index, row in slider_df.iterrows():
        output_df.append({
            "DATEINAME": f"b-{field.replace('_','-')}",
            "RAUM": "Münster",
            "QUELLE_INSTITUTION": "Grünflächenamt",
            "THEMENBEREICH": themenbereich,
            "MERKMAL": f"{row['gruppe']} - {fields[field][0]}",
            "ZEIT": row['jahr'],
            "WERT": int(row[field]),
            "WERTEEINHEIT": fields[field][1]
        })

for pfl in pflanzungen:
    output_df.append({
        "DATEINAME": "b-pflanzungen",
        "RAUM": "Münster",
        "QUELLE_INSTITUTION": "Grünflächenamt",
        "THEMENBEREICH": themenbereich + 1,
        "MERKMAL": "Pflanzungen",
        "ZEIT": pfl,
        "WERT": int(pflanzungen[pfl]),
        "WERTEEINHEIT": "Anzahl"
    })

for jahr, altersverteilung in altersverteilungen.items():
    for altersklasse, anzahl in altersverteilung.items():
        output_df.append({
            "DATEINAME": "b-alter",
            "RAUM": "Münster",
            "QUELLE_INSTITUTION": "Grünflächenamt",
            "THEMENBEREICH": themenbereich + 2,
            "MERKMAL": altersklasse,
            "ZEIT": jahr,
            "WERT": int(anzahl),
            "WERTEEINHEIT": "Anzahl",
        })

for jahr, altersverteilung in altersverteilungen.items():
    output_df.append({
        "DATEINAME": "b-anzahl-baeume",
        "RAUM": "Münster",
        "QUELLE_INSTITUTION": "Grünflächenamt",
        "THEMENBEREICH": themenbereich + 3,
        "MERKMAL": "Gesamtanzahl Bäume",
        "ZEIT": jahr,
        "WERT": int(altersverteilung.sum()),
        "WERTEEINHEIT": "Anzahl",
    })

output_dfdf = pd.DataFrame(output_df)

output_dfdf.to_csv("data/klimadashboard-baeume.csv", index=False, sep=";")
